## 1. Hugging Face Login

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

## 2. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

## 4. Load Tokenizer

In [ ]:
from collections import defaultdict, Counter
import heapq

SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class BytelevelBPE:

    def __init__(self, vocab_size=5000):
        self.vocab_size = vocab_size
        self.specials = SPECIALS.copy()
        self.byte_vocab = [str(i) for i in range(256)]
        self.merges = []
        self.merge_ranks = {}
        self.vocab = []
        self.stoi = {}
        self.itos = {}
        # Maps each BPE symbol to the bytes represented by it
        self.symbol_to_bytes = {}
    
    # 1. WORD -> UTF-8 BYTE SYMBOLS

    def word_to_symbols(self, word):
        byte_seq = word.encode("utf-8")
        return [str(b) for b in byte_seq] + ["</w>"]

    # 2. BUILD WORD FREQUENCIES

    def build_word_freqs(self, corpus):
        word_counter = Counter()
        print("Building word frequencies...")
        for text in corpus:
            if not isinstance(text, str):
                continue

            word_counter.update(text.split())
        print(
            f"Unique whitespace-separated words: "
            f"{len(word_counter):,}"
        )
        # Convert each unique word to byte symbols only once
        word_freqs = {}

        for word, freq in word_counter.items():
            symbols = tuple(self.word_to_symbols(word))
            word_freqs[symbols] = (
                word_freqs.get(symbols, 0) + freq
            )

        return word_freqs


    # 3. PAIRS OCCURRING IN ONE WORD

    @staticmethod
    def get_pairs(symbols):
        return {
            (symbols[i], symbols[i + 1])
            for i in range(len(symbols) - 1)
        }

    # 4. COUNT PAIR OCCURRENCES IN ONE WORD

    @staticmethod
    def pair_counts_in_word(symbols):
        counts = Counter()
        for i in range(len(symbols) - 1):
            counts[
                (symbols[i], symbols[i + 1])
            ] += 1

        return counts

    # 5. MERGE A PAIR INSIDE ONE WORD

    @staticmethod
    def merge_symbols(symbols, pair, merged_symbol):
        new_symbols = []
        i = 0
        while i < len(symbols):
            if (
                i < len(symbols) - 1
                and symbols[i] == pair[0]
                and symbols[i + 1] == pair[1]
            ):
                new_symbols.append(merged_symbol)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1

        return tuple(new_symbols)

    # 6. TRAIN OPTIMIZED BPE

    def train(self, corpus):
        word_freqs = self.build_word_freqs(corpus)
        vocab = (
            self.specials
            + self.byte_vocab
            + ["</w>"]
        )
        seen = set(vocab)
        for b in range(256):
            self.symbol_to_bytes[str(b)] = bytes([b])
        self.symbol_to_bytes["</w>"] = b""

        words = []
        frequencies = []
        for symbols, freq in word_freqs.items():
            words.append(symbols)
            frequencies.append(freq)
        del word_freqs

        pair_counts = defaultdict(int)
        pair_to_words = defaultdict(set)

        print("Building initial pair statistics...")

        for word_id, symbols in enumerate(words):
            freq = frequencies[word_id]
            local_counts = self.pair_counts_in_word(symbols)
            for pair, count in local_counts.items():
                pair_counts[pair] += count * freq
                pair_to_words[pair].add(word_id)

        print(
            f"Initial unique pairs: "
            f"{len(pair_counts):,}"
        )

        # MAX HEAP
        #
        # Python has min-heap, so use negative frequency.
        #
        # We keep stale entries and validate them when popped.
        # This avoids rebuilding the heap after every merge.

        heap = []
        for pair, count in pair_counts.items():
            heapq.heappush(heap,(-count, pair))
        target_merges = max(0,self.vocab_size - len(vocab))
        print(f"Target merges: {target_merges:,}")
        while (len(vocab) < self.vocab_size and heap):
            while heap:
                neg_count, pair = heapq.heappop(heap)
                heap_count = -neg_count
                current_count = pair_counts.get(pair,0)
                # Ignore stale heap entries
                if (current_count == heap_count and current_count > 0):
                    break
            else:
                break

            max_freq = pair_counts.get(pair, 0)

            # Same stopping condition as before
            if max_freq <= 1:
                break

            # Create collision-free merged symbol
            merged_symbol = (f"<bpe_{len(self.merges)}>")

            # Byte representation of new symbol
            left_bytes = self.symbol_to_bytes[pair[0]]
            right_bytes = self.symbol_to_bytes[pair[1]]
            self.symbol_to_bytes[merged_symbol] = left_bytes + right_bytes

            self.merges.append(pair)
            self.merge_ranks[pair] = (len(self.merges) - 1,merged_symbol)
            if merged_symbol not in seen:
                seen.add(merged_symbol)
                vocab.append(merged_symbol)

            # ----------------------------------------------------
            # IMPORTANT OPTIMIZATION
            #
            # Only process words containing selected pair.
            # ----------------------------------------------------

            affected_word_ids = list(pair_to_words.get(pair, set()))
            changed_pairs = set()
            for word_id in affected_word_ids:
                old_symbols = words[word_id]
                freq = frequencies[word_id]
                # Pair may have disappeared because of an earlier
                # update involving this word
                if pair not in self.get_pairs(old_symbols):
                    continue
                
                # Statistics BEFORE merge
                old_counts = self.pair_counts_in_word(old_symbols)

                # Merge selected pair
                new_symbols = self.merge_symbols(old_symbols,pair,merged_symbol)
                if new_symbols == old_symbols:
                    continue

                # Statistics AFTER merge
                new_counts = self.pair_counts_in_word(new_symbols)

                # Save updated word
                words[word_id] = new_symbols

                all_pairs = (
                    set(old_counts)
                    | set(new_counts)
                )

                # Update only pairs affected by this word

                for p in all_pairs:

                    old_n = old_counts.get(p, 0)
                    new_n = new_counts.get(p, 0)
                    delta = (
                        new_n - old_n
                    ) * freq

                    if delta != 0:
                        pair_counts[p] += delta
                        changed_pairs.add(p)

                    # Update inverted index
                    if new_n > 0:
                        pair_to_words[p].add(word_id)
                    else:
                        pair_to_words[p].discard(word_id)

            # Selected pair should now be gone from affected words
            pair_to_words[pair].clear()

            # Push updated frequencies into heap
            for p in changed_pairs:
                count = pair_counts.get(p, 0)
                if count > 0:
                    heapq.heappush(heap,(-count, p))

        self.vocab = vocab
        self.vocab_size = len(vocab)
        self.stoi = {
            symbol: i
            for i, symbol in enumerate(vocab)
        }

        self.itos = {
            i: symbol
            for symbol, i in self.stoi.items()
        }

        print("\nTokenizer training complete.")

        print(
            "Final vocabulary size:",
            self.vocab_size
        )

        print(
            "Number of merges:",
            len(self.merges)
        )

    # 7. APPLY BPE TO ONE WORD

    def apply_bpe(self, word):
        symbols = self.word_to_symbols(word)
        while len(symbols) > 1:
            best_pair = None
            best_rank = float("inf")
            best_merged_symbol = None

            for i in range(len(symbols) - 1):
                pair = (symbols[i],symbols[i + 1])
                merge_info = self.merge_ranks.get(pair)
                if merge_info is not None:
                    rank, merged_symbol = merge_info
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = pair
                        best_merged_symbol = merged_symbol

            if best_pair is None:
                break

            symbols = list(
                self.merge_symbols(
                    tuple(symbols),
                    best_pair,
                    best_merged_symbol
                )
            )

        return symbols

    # 8. ENCODE

    def encode(self,text,add_bos=False,add_eos=False):
        ids = []
        if add_bos:
            ids.append(
                self.stoi["<s>"]
            )

        for word in text.split():
            symbols = self.apply_bpe(word)
            for symbol in symbols:
                ids.append(
                    self.stoi.get(
                        symbol,
                        self.stoi["<unk>"]
                    )
                )

        if add_eos:
            ids.append(
                self.stoi["</s>"]
            )

        return ids

    # 9. DECODE

    def decode(self, ids):

        words = []
        current_bytes = bytearray()

        for token_id in ids:
            symbol = self.itos.get(token_id)
            if symbol is None:
                continue
            if symbol in self.specials:
                continue

            # Determine whether this token ends a word
            ends_word = self._symbol_ends_word(symbol)
            byte_value = self.symbol_to_bytes.get(symbol, b"")
            current_bytes.extend(byte_value)

            if ends_word:
                try:
                    word = bytes(current_bytes).decode("utf-8")
                    words.append(word)
                except UnicodeDecodeError:
                    words.append("<unk>")
                current_bytes = bytearray()

        # Remaining bytes
        if current_bytes:
            try:
                words.append(bytes(current_bytes).decode("utf-8"))
            except UnicodeDecodeError:
                words.append("<unk>")

        return " ".join(words)

    # 10. CHECK WHETHER SYMBOL CONTAINS </w> AT END

    def _symbol_ends_word(self, symbol):
        if symbol == "</w>":
            return True
        # Find merge that created this symbol
        for pair, (_, merged) in self.merge_ranks.items():
            if merged == symbol:
                return self._symbol_ends_word(
                    pair[1]
                )

        return False

    # 11. VOCAB SIZE

    def get_vocab_size(self):
        return len(self.vocab)

## 9. Define The Model

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        if self.d_head % 2 != 0:
            raise ValueError("RoPE requires an even head dimension.")

        inv_freq = 1.0 / (
            10000.0 ** (torch.arange(0, self.d_head, 2).float() / self.d_head)
        )
        angles = torch.outer(torch.arange(max_seq_len).float(), inv_freq)
        self.register_buffer("rope_cos", angles.cos(), persistent=False)
        self.register_buffer("rope_sin", angles.sin(), persistent=False)
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

    def apply_rope(self, q, k, offset=0):
        T = q.size(2)
        cos = self.rope_cos[offset:offset + T][None, None].to(q.dtype)
        sin = self.rope_sin[offset:offset + T][None, None].to(q.dtype)

        def rotate(x):
            first, second = x.chunk(2, dim=-1)
            return torch.cat(
                [first * cos - second * sin, first * sin + second * cos],
                dim=-1,
            )

        return rotate(q), rotate(k)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d_model) -> (B, T, d_model)"""
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        q, k = self.apply_rope(q, k)

        scores = q @ k.transpose(-2, -1) * self.attention_scale

        scores = scores.masked_fill(self.causal_mask[:T, :T] == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)

        weighted_sum = attn_weights @ v
        weighted_sum = weighted_sum.transpose(1, 2).contiguous().view(B, T, C)

        out = self.out_proj(weighted_sum)
        out = self.resid_dropout(out)

        return out

In [ ]:
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 512
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4          
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "rope"   
    attention_scale: float = None      
    norm_first: bool = False            # norm_first=True is for pre-norm else post-norm


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        # TODO: two linear layers with a GELU in between, plus dropout
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout,
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

        self.norm_first = cfg.norm_first

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))
        else:
            x = self.ln1(x + self.attn(x))
            x = self.ln2(x + self.ffn(x))
        return x


class DecoderLM(nn.Module):
    """The model class the autograder imports directly. Do not rename it."""

    def __init__(self, cfg: LMConfig):
        super().__init__()
        if cfg.pos_emb_type != "rope":
            raise ValueError("This notebook requires a RoPE model/checkpoint.")
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)

        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """idx: (B, T) LongTensor of token ids -> logits (B, T, vocab_size)."""
        B, T = idx.shape

        x = self.tok_emb(idx)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.head(x)
        return logits


# **Phase 7**

In [ ]:
import gc
import json
import math
import random
import shutil
import zipfile

from pathlib import Path
from functools import lru_cache
from types import MethodType

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.nn.utils.rnn import pad_sequence
from torch.utils.checkpoint import checkpoint

from sklearn.feature_extraction.text import TfidfVectorizer
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient


HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

# Original Phase 1 Model 1 files.
MODEL_DIR = Path(
    "/kaggle/input/datasets/tejasvaidya137/model1-for-phase7-files"
)

OUTPUT_DIR = Path(
    "/kaggle/working/phase7_model1_fresh_lora"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42

# LoRA settings
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

# Training settings
RETRIEVER_EPOCHS = 5
RERANKER_EPOCHS = 3
LEARNING_RATE = 1e-4
PATIENCE = 2
QUERY_BATCH_SIZE = 8

# Retriever settings
RETRIEVAL_LENGTH = 1024
MAX_CHUNKS = 8
CHUNK_OVERLAP = 64
CHUNK_BATCH_SIZE = 8
PROJECTION_DIM = 256
TEMPERATURE = 0.07

# Training candidates per query
POSITIVES = 2
HARD_NEGATIVES = 2
RANDOM_NEGATIVES = 1

# First-stage retrieval
RETRIEVE_TOP_K = 100
RRF_K = 60
NEURAL_WEIGHTS = [0.0, 0.25, 0.5, 0.75, 1.0]

# Reranker settings
PAIR_LENGTH = 1024
PASSAGE_OVERLAP = 64
PASSAGE_VIEWS = 2
PAIR_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8

# Use None to rerank the complete union of both top-100 lists.
RERANK_TOP_K = 50

FUSION_WEIGHTS = [0.0, 0.25, 0.5, 0.75, 1.0]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

use_amp = device.type == "cuda"

amp_dtype = (
    torch.bfloat16
    if use_amp and torch.cuda.is_bf16_supported()
    else torch.float16
)


def amp_context():
    return torch.autocast(
        device_type=device.type,
        dtype=amp_dtype,
        enabled=use_amp,
    )


def load_json(path):
    with open(path, encoding="utf-8") as file:
        return json.load(file)

In [ ]:
def load_model1():
    saved = torch.load(
        MODEL_DIR / "model.pt",
        map_location="cpu",
        weights_only=True,
    )

    model = DecoderLM(
        LMConfig(**saved["model_config"])
    )

    model.load_state_dict(
        saved["model_state_dict"],
        strict=True,
    )

    return model


state = load_json(MODEL_DIR / "tokenizer.json")

tokenizer = BytelevelBPE(state["vocab_size"])
tokenizer.specials = state["specials"]
tokenizer.vocab = state["vocab"]
tokenizer.vocab_size = len(tokenizer.vocab)

tokenizer.stoi = {
    symbol: i
    for i, symbol in enumerate(tokenizer.vocab)
}

tokenizer.itos = {
    i: symbol
    for symbol, i in tokenizer.stoi.items()
}

tokenizer.merges = [
    tuple(pair) for pair in state["merges"]
]

tokenizer.merge_ranks = {
    tuple(row["pair"]): (
        row["rank"],
        row["merged_symbol"],
    )
    for row in state["merge_ranks"]
}

tokenizer.symbol_to_bytes = {
    symbol: bytes(value)
    for symbol, value in state["symbol_to_bytes"].items()
}

tokenizer.apply_bpe = lru_cache(maxsize=100000)(
    tokenizer.apply_bpe
)

del state

PAD_ID = tokenizer.stoi["<pad>"]
BOS_ID = tokenizer.stoi["<s>"]
EOS_ID = tokenizer.stoi["</s>"]


def fast_attention(self, x):
    batch, length, width = x.shape

    q = self.q_proj(x).view(
        batch, length, self.n_heads, self.d_head
    ).transpose(1, 2)

    k = self.k_proj(x).view(
        batch, length, self.n_heads, self.d_head
    ).transpose(1, 2)

    v = self.v_proj(x).view(
        batch, length, self.n_heads, self.d_head
    ).transpose(1, 2)

    q, k = self.apply_rope(q, k)

    output = F.scaled_dot_product_attention(
        q,
        k,
        v,
        is_causal=True,
        scale=self.attention_scale,
        dropout_p=(
            self.attn_dropout.p if self.training else 0.0
        ),
    )

    output = output.transpose(1, 2).contiguous()
    output = output.view(batch, length, width)

    return self.resid_dropout(
        self.out_proj(output)
    )


def patch_attention(model):
    for block in model.blocks:
        block.attn.forward = MethodType(
            fast_attention,
            block.attn,
        )


def pack_sequences(sequences):
    lengths = torch.tensor(
        [len(sequence) for sequence in sequences],
        device=device,
    )

    ids = pad_sequence(
        [
            torch.tensor(sequence, dtype=torch.long)
            for sequence in sequences
        ],
        batch_first=True,
        padding_value=PAD_ID,
    ).to(device)

    return ids, lengths

In [ ]:
def read_cases(path, labeled=False):
    content = load_json(path)

    if isinstance(content, dict):
        items = content.items()
    else:
        items = [
            (
                row.get(
                    "id",
                    row.get("query_id", row.get("case_id")),
                ),
                row,
            )
            for row in content
        ]

    records = {}

    for case_id, row in items:
        if case_id is None:
            raise ValueError("Missing case ID.")

        if not isinstance(row, dict):
            row = {"text": row}

        text = row["text"]

        if isinstance(text, list):
            text = " ".join(
                part if isinstance(part, str) else part["text"]
                for part in text
            )

        records[str(case_id)] = {"text": text}

        if labeled:
            records[str(case_id)]["relevant_candidates"] = {
                str(candidate_id)
                for candidate_id in row["relevant_candidates"]
            }

    return records


data = {}
tokens = {}

for split in ["train", "dev", "test"]:
    records = []

    for filename in [
        f"{split}.json",
        f"{split}_candidates.json",
    ]:
        path = hf_hub_download(
            repo_id="Exploration-Lab/CS-781-Capstone",
            repo_type="dataset",
            filename=f"Phase_7_PCR/{split.title()}/{filename}",
            token=HF_TOKEN,
        )

        labeled = (
            split != "test"
            and filename == f"{split}.json"
        )

        records.append(
            read_cases(path, labeled=labeled)
        )

    # Each split contains (queries, candidates).
    data[split] = tuple(records)

    tokens[split] = tuple(
        {
            case_id: tokenizer.encode(row["text"])
            for case_id, row in group.items()
        }
        for group in records
    )

    print(
        split,
        len(records[0]), "queries,",
        len(records[1]), "candidates",
    )


train_queries, train_candidates = data["train"]
dev_queries, dev_candidates = data["dev"]
test_queries, test_candidates = data["test"]

for queries, candidates in [data["train"], data["dev"]]:
    for row in queries.values():
        assert row["relevant_candidates"] <= set(candidates)

In [ ]:
class LoRALinear(nn.Module):
    def __init__(
        self,
        linear,
        rank=LORA_R,
        alpha=LORA_ALPHA,
    ):
        super().__init__()

        self.base = linear
        self.base.requires_grad_(False)

        self.A = nn.Linear(
            linear.in_features,
            rank,
            bias=False,
        )

        self.B = nn.Linear(
            rank,
            linear.out_features,
            bias=False,
        )

        nn.init.kaiming_uniform_(
            self.A.weight,
            a=math.sqrt(5),
        )
        nn.init.zeros_(self.B.weight)

        self.dropout = nn.Dropout(LORA_DROPOUT)
        self.scale = alpha / rank

    def forward(self, x):
        update = self.B(
            self.A(self.dropout(x))
        )

        return self.base(x) + update * self.scale


class Phase7Model(nn.Module):
    def __init__(self, task):
        super().__init__()

        base = load_model1()
        patch_attention(base)

        self.cfg = base.cfg
        self.task = task

        self.tok_emb = base.tok_emb
        self.blocks = base.blocks
        self.ln_f = base.ln_f

        # Freeze the original Model 1 parameters.
        self.requires_grad_(False)

        for block in self.blocks:
            block.attn.q_proj = LoRALinear(
                block.attn.q_proj
            )

            block.attn.v_proj = LoRALinear(
                block.attn.v_proj
            )

        output_size = (
            PROJECTION_DIM if task == "retriever" else 1
        )

        self.head = nn.Linear(
            base.cfg.d_model,
            output_size,
            bias=False,
        )

    def forward(self, ids, lengths):
        x = self.tok_emb(ids)

        for block in self.blocks:
            if self.training:
                x = checkpoint(
                    block,
                    x,
                    use_reentrant=False,
                )
            else:
                x = block(x)

        rows = torch.arange(
            len(ids),
            device=ids.device,
        )

        # Final real EOS token, excluding padding.
        return self.ln_f(
            x[rows, lengths - 1]
        )


def save_model(model, path, epoch, result):
    trainable_state = {
        name: parameter.detach().cpu().clone()
        for name, parameter in model.named_parameters()
        if parameter.requires_grad
    }

    torch.save(
        {
            "task": model.task,
            "epoch": epoch,
            "result": result,
            "trainable_state": trainable_state,

            "settings": {
                "lora_r": LORA_R,
                "lora_alpha": LORA_ALPHA,
                "lora_dropout": LORA_DROPOUT,
                "projection_dim": PROJECTION_DIM,
            },
        },
        path,
    )


def load_phase7(path):
    saved = torch.load(
        path,
        map_location="cpu",
        weights_only=True,
    )

    expected_settings = {
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "projection_dim": PROJECTION_DIM,
    }

    assert saved["settings"] == expected_settings

    model = Phase7Model(saved["task"])

    expected_parameters = {
        name
        for name, parameter in model.named_parameters()
        if parameter.requires_grad
    }

    assert (
        set(saved["trainable_state"])
        == expected_parameters
    )

    # Original frozen weights are already loaded.
    model.load_state_dict(
        saved["trainable_state"],
        strict=False,
    )

    return model.to(device).eval(), saved


retriever = Phase7Model("retriever").to(device)

assert retriever.cfg.vocab_size == tokenizer.vocab_size

RETRIEVAL_LENGTH = min(
    RETRIEVAL_LENGTH,
    retriever.cfg.max_seq_len,
)

PAIR_LENGTH = min(
    PAIR_LENGTH,
    retriever.cfg.max_seq_len,
)

PASSAGE_TOKENS = (PAIR_LENGTH - 3) // 2

assert 0 <= CHUNK_OVERLAP < RETRIEVAL_LENGTH - 2
assert 0 <= PASSAGE_OVERLAP < PASSAGE_TOKENS

print(
    "Retriever trainable parameters:",
    sum(
        parameter.numel()
        for parameter in retriever.parameters()
        if parameter.requires_grad
    ),
)

In [ ]:
def split_tokens(ids, size, overlap):
    parts = []

    for start in range(0, len(ids), size - overlap):
        parts.append(ids[start:start + size])

        if start + size >= len(ids):
            break

    return parts or [[]]


def retrieval_chunks(ids):
    parts = split_tokens(
        ids,
        RETRIEVAL_LENGTH - 2,
        CHUNK_OVERLAP,
    )

    if len(parts) > MAX_CHUNKS:
        positions = np.linspace(
            0,
            len(parts) - 1,
            MAX_CHUNKS,
            dtype=int,
        )

        parts = [parts[i] for i in positions]

    return [
        [BOS_ID] + part + [EOS_ID]
        for part in parts
    ]


chunks = {}

for split in tokens:
    chunks[split] = tuple(
        {
            case_id: retrieval_chunks(ids)
            for case_id, ids in group.items()
        }
        for group in tokens[split]
    )


def lexical_rankings(split):
    queries, candidates = data[split]
    candidate_ids = list(candidates)

    tfidf = TfidfVectorizer(
        ngram_range=(1, 2),
        sublinear_tf=True,
        max_features=60000,
        dtype=np.float32,
    )

    matrix = tfidf.fit_transform(
        [candidates[c]["text"] for c in candidate_ids]
    )

    rankings = {}

    for query_id, row in queries.items():
        query_vector = tfidf.transform([row["text"]])

        scores = (
            query_vector @ matrix.T
        ).toarray().ravel()

        order = np.argsort(
            -scores,
            kind="stable",
        )[:RETRIEVE_TOP_K]

        rankings[query_id] = [
            candidate_ids[i] for i in order
        ]

    return rankings


lexical = {
    split: lexical_rankings(split)
    for split in ["train", "dev"]
}


def encode_documents(model, documents):
    sequences = []
    owners = []

    for owner, document in enumerate(documents):
        sequences.extend(document)
        owners.extend([owner] * len(document))

    vectors = []

    for start in range(0, len(sequences), CHUNK_BATCH_SIZE):
        batch = sequences[start:start + CHUNK_BATCH_SIZE]

        vectors.append(
            model(*pack_sequences(batch)).float()
        )

    vectors = torch.cat(vectors)

    owners = torch.tensor(
        owners,
        device=device,
    )

    totals = vectors.new_zeros(
        len(documents),
        vectors.shape[1],
    ).index_add(
        0,
        owners,
        vectors,
    )

    counts = torch.bincount(
        owners,
        minlength=len(documents),
    ).unsqueeze(1)

    document_vectors = totals / counts

    return F.normalize(
        model.head(document_vectors).float(),
        dim=-1,
    )


@torch.inference_mode()
def make_index(model, records):
    model.eval()

    ids = list(records)
    vectors = []

    for start in range(0, len(ids), EVAL_BATCH_SIZE):
        batch_ids = ids[start:start + EVAL_BATCH_SIZE]

        with amp_context():
            batch_vectors = encode_documents(
                model,
                [records[case_id] for case_id in batch_ids],
            )

        vectors.append(batch_vectors.cpu())

    return ids, torch.cat(vectors)


def neural_rankings(model, split):
    query_ids, query_vectors = make_index(
        model,
        chunks[split][0],
    )

    candidate_ids, candidate_vectors = make_index(
        model,
        chunks[split][1],
    )

    scores = query_vectors @ candidate_vectors.T
    rankings = {}

    for query_id, row in zip(query_ids, scores):
        order = torch.argsort(
            row,
            descending=True,
            stable=True,
        ).tolist()[:RETRIEVE_TOP_K]

        rankings[query_id] = [
            candidate_ids[i] for i in order
        ]

    return rankings


def fuse_rankings(first, second, weight):
    fused = {}

    for query_id in first:
        scores = {}

        for ranking, source_weight in [
            (first[query_id], weight),
            (second[query_id], 1.0 - weight),
        ]:
            for rank, candidate_id in enumerate(ranking, 1):
                scores[candidate_id] = (
                    scores.get(candidate_id, 0.0)
                    + source_weight / (RRF_K + rank)
                )

        fused[query_id] = sorted(
            scores,
            key=lambda c: (-scores[c], c),
        )

    return fused


def retrieval_scores(rankings, queries):
    assert set(rankings) == set(queries)

    hits = sum(
        len(
            set(rankings[query_id][:7])
            & row["relevant_candidates"]
        )
        for query_id, row in queries.items()
    )

    gold = sum(
        len(row["relevant_candidates"])
        for row in queries.values()
    )

    predicted = 7 * len(queries)

    return {
        "precision": hits / predicted,
        "recall": hits / max(gold, 1),
        "f1": 2 * hits / (predicted + gold),
    }

In [ ]:
training_ids = [
    query_id
    for query_id, row in train_queries.items()
    if (
        0 < len(row["relevant_candidates"])
        < len(train_candidates)
    )
]

train_candidate_ids = list(train_candidates)

assert training_ids


def sample_candidates(query_id, rankings):
    gold = train_queries[query_id]["relevant_candidates"]

    positives = random.sample(
        sorted(gold),
        min(POSITIVES, len(gold)),
    )

    hard_pool = [
        candidate_id
        for candidate_id in rankings[query_id]
        if candidate_id not in gold
    ][:50]

    negatives = random.sample(
        hard_pool,
        min(HARD_NEGATIVES, len(hard_pool)),
    )

    other = [
        candidate_id
        for candidate_id in train_candidate_ids
        if candidate_id not in gold
        and candidate_id not in negatives
    ]

    negatives += random.sample(
        other,
        min(RANDOM_NEGATIVES, len(other)),
    )

    assert negatives

    return positives + negatives, len(positives)


def retriever_loss(model, query_ids):
    candidates = []

    for query_id in query_ids:
        sampled, _ = sample_candidates(
            query_id,
            lexical["train"],
        )
        candidates.extend(sampled)

    candidates = list(dict.fromkeys(candidates))

    query_vectors = encode_documents(
        model,
        [chunks["train"][0][q] for q in query_ids],
    )

    candidate_vectors = encode_documents(
        model,
        [chunks["train"][1][c] for c in candidates],
    )

    scores = (
        query_vectors @ candidate_vectors.T
    ) / TEMPERATURE

    positive_mask = torch.tensor(
        [
            [
                candidate_id
                in train_queries[query_id]["relevant_candidates"]
                for candidate_id in candidates
            ]
            for query_id in query_ids
        ],
        device=device,
        dtype=torch.bool,
    )

    log_probs = F.log_softmax(
        scores.float(),
        dim=-1,
    )

    loss_per_query = -(
        log_probs * positive_mask
    ).sum(1) / positive_mask.sum(1)

    return loss_per_query.mean()


def evaluate_retriever(model):
    rankings = neural_rankings(model, "dev")

    return retrieval_scores(
        rankings,
        dev_queries,
    )


def train_stage(
    model,
    loss_function,
    evaluate,
    epochs,
    filename,
    baseline=None,
):
    parameters = [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ]

    optimizer = torch.optim.AdamW(
        parameters,
        lr=LEARNING_RATE,
        weight_decay=0.01,
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(
            use_amp and amp_dtype == torch.float16
        ),
    )

    path = OUTPUT_DIR / filename

    best_f1 = (
        -1.0 if baseline is None else baseline["f1"]
    )

    if baseline is not None:
        save_model(model, path, 0, baseline)

    history = []
    bad_epochs = 0

    for epoch in range(1, epochs + 1):
        model.train()

        order = training_ids.copy()
        random.shuffle(order)

        total_loss = 0.0

        batches = math.ceil(
            len(order) / QUERY_BATCH_SIZE
        )

        for step, start in enumerate(
            range(0, len(order), QUERY_BATCH_SIZE),
            1,
        ):
            batch = order[
                start:start + QUERY_BATCH_SIZE
            ]

            optimizer.zero_grad(set_to_none=True)

            with amp_context():
                loss = loss_function(model, batch)

            if not torch.isfinite(loss):
                raise RuntimeError(
                    "Non-finite training loss."
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                parameters,
                1.0,
            )

            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item() * len(batch)

            if step % 100 == 0 or step == batches:
                print(
                    f"{model.task} | "
                    f"Epoch {epoch} | "
                    f"Batch {step}/{batches} | "
                    f"Loss {loss.item():.4f}",
                    flush=True,
                )

        result = evaluate(model)

        record = {
            "epoch": epoch,
            "train_loss": total_loss / len(order),
            **result,
        }

        history.append(record)
        print(record)

        history_path = (
            OUTPUT_DIR / f"{model.task}_history.json"
        )

        history_path.write_text(
            json.dumps(history, indent=2)
        )

        if result["f1"] > best_f1:
            best_f1 = result["f1"]
            bad_epochs = 0

            save_model(
                model,
                path,
                epoch,
                result,
            )

            print("Saved best checkpoint.")

        else:
            bad_epochs += 1

            if bad_epochs >= PATIENCE:
                print("Early stopping.")
                break

    return history


retriever_history = train_stage(
    retriever,
    retriever_loss,
    evaluate_retriever,
    RETRIEVER_EPOCHS,
    "best_retriever.pt",
)

del retriever
gc.collect()

if use_amp:
    torch.cuda.empty_cache()

retriever, retriever_saved = load_phase7(
    OUTPUT_DIR / "best_retriever.pt"
)

print(
    "Best retriever:",
    retriever_saved["result"],
)

In [ ]:
dev_neural = neural_rankings(
    retriever,
    "dev",
)

results = []

for weight in NEURAL_WEIGHTS:
    rankings = fuse_rankings(
        dev_neural,
        lexical["dev"],
        weight,
    )

    result = {
        "neural_weight": weight,
        **retrieval_scores(rankings, dev_queries),
    }

    results.append(result)
    print(result)

hybrid_best = max(
    results,
    key=lambda row: row["f1"],
)

NEURAL_WEIGHT = hybrid_best["neural_weight"]

print("Selected hybrid result:", hybrid_best)

shortlists = {
    "dev": fuse_rankings(
        dev_neural,
        lexical["dev"],
        NEURAL_WEIGHT,
    )
}

train_neural = neural_rankings(
    retriever,
    "train",
)

shortlists["train"] = fuse_rankings(
    train_neural,
    lexical["train"],
    NEURAL_WEIGHT,
)

# The retriever stays fixed during reranker training.
retriever.cpu()

if use_amp:
    torch.cuda.empty_cache()

assert RERANK_TOP_K is None or RERANK_TOP_K >= 7


for k in [30, 50, 100, None]:
    hits = 0
    gold_total = 0
    possible_hits = 0

    for query_id, row in dev_queries.items():
        shortlist = shortlists["dev"][query_id][:k]

        available = len(
            set(shortlist)
            & row["relevant_candidates"]
        )

        hits += available
        gold_total += len(row["relevant_candidates"])
        possible_hits += min(7, available)

    print({
        "shortlist": k or "full union",
        "recall": hits / max(gold_total, 1),
        "oracle_f1": (
            2 * possible_hits
            / (7 * len(dev_queries) + gold_total)
        ),
    })

In [ ]:
passages = {}

for split in tokens:
    passages[split] = tuple(
        {
            case_id: split_tokens(
                ids,
                PASSAGE_TOKENS,
                PASSAGE_OVERLAP,
            )
            for case_id, ids in group.items()
        }
        for group in tokens[split]
    )


def token_terms(ids):
    return [str(token_id) for token_id in ids]


passage_tfidf = TfidfVectorizer(
    analyzer=token_terms,
    sublinear_tf=True,
    dtype=np.float32,
)

passage_tfidf.fit(
    part
    for group in passages["train"]
    for document_parts in group.values()
    for part in document_parts
)

passage_vectors = {}

for split in passages:
    passage_vectors[split] = tuple(
        {
            case_id: passage_tfidf.transform(parts)
            for case_id, parts in group.items()
        }
        for group in passages[split]
    )


@lru_cache(maxsize=100000)
def select_passages(split, query_id, candidate_id):
    query_matrix = passage_vectors[split][0][query_id]
    candidate_matrix = passage_vectors[split][1][candidate_id]

    scores = (
        query_matrix @ candidate_matrix.T
    ).toarray()

    selected = []

    number_of_views = min(
        PASSAGE_VIEWS,
        *scores.shape,
    )

    for _ in range(number_of_views):
        query_index, candidate_index = np.unravel_index(
            scores.argmax(),
            scores.shape,
        )

        selected.append(
            (int(query_index), int(candidate_index))
        )

        # Avoid repeating the same passage in another view.
        scores[query_index, :] = -np.inf
        scores[:, candidate_index] = -np.inf

    return tuple(selected)


def score_pairs(
    model,
    split,
    query_id,
    candidates,
    batch_size,
):
    sequences = []
    owners = []

    for owner, candidate_id in enumerate(candidates):
        selected = select_passages(
            split,
            query_id,
            candidate_id,
        )

        for query_index, candidate_index in selected:
            query_part = passages[split][0][query_id][
                query_index
            ]

            candidate_part = passages[split][1][candidate_id][
                candidate_index
            ]

            sequence = (
                [BOS_ID]
                + query_part
                + [EOS_ID]
                + candidate_part
                + [EOS_ID]
            )

            sequences.append(sequence)
            owners.append(owner)

    scores = []

    for start in range(0, len(sequences), batch_size):
        batch = sequences[start:start + batch_size]

        hidden = model(
            *pack_sequences(batch)
        )

        scores.append(
            model.head(hidden).squeeze(-1).float()
        )

    scores = torch.cat(scores)

    owners = torch.tensor(
        owners,
        device=device,
    )

    totals = scores.new_zeros(
        len(candidates)
    ).index_add(
        0,
        owners,
        scores,
    )

    counts = torch.bincount(
        owners,
        minlength=len(candidates),
    )

    # Average each candidate's passage-pair scores.
    return totals / counts

In [ ]:
def reranker_loss(model, query_ids):
    losses = []

    for query_id in query_ids:
        candidates, positive_count = sample_candidates(
            query_id,
            shortlists["train"],
        )

        scores = score_pairs(
            model,
            "train",
            query_id,
            candidates,
            PAIR_BATCH_SIZE,
        )

        positive_scores = scores[:positive_count]
        negative_scores = scores[positive_count:]

        differences = (
            negative_scores[None, :]
            - positive_scores[:, None]
        )

        losses.append(
            F.softplus(differences).mean()
        )

    return torch.stack(losses).mean()


@torch.inference_mode()
def rerank(model, split):
    model.eval()
    rankings = {}

    for step, (query_id, ids) in enumerate(
        shortlists[split].items(),
        1,
    ):
        candidates = ids[:RERANK_TOP_K]

        with amp_context():
            values = score_pairs(
                model,
                split,
                query_id,
                candidates,
                EVAL_BATCH_SIZE,
            ).cpu().tolist()

        order = sorted(
            range(len(candidates)),
            key=lambda i: (-values[i], i),
        )

        rankings[query_id] = [
            candidates[i] for i in order
        ]

        if step % 25 == 0:
            print(
                f"{split}: "
                f"{step}/{len(shortlists[split])} "
                "queries reranked",
                flush=True,
            )

    return rankings


def combine_reranker(rankings, split, weight):
    if weight == 0:
        return shortlists[split]

    original = {
        query_id: ids[:RERANK_TOP_K]
        for query_id, ids in shortlists[split].items()
    }

    return fuse_rankings(
        rankings,
        original,
        weight,
    )


def evaluate_reranker(model):
    rankings = rerank(model, "dev")
    results = []

    for weight in FUSION_WEIGHTS:
        fused = combine_reranker(
            rankings,
            "dev",
            weight,
        )

        results.append({
            "reranker_weight": weight,
            **retrieval_scores(fused, dev_queries),
        })

    return max(
        results,
        key=lambda row: row["f1"],
    )


reranker = Phase7Model("reranker").to(device)

baseline = {
    "reranker_weight": 0.0,
    **retrieval_scores(
        shortlists["dev"],
        dev_queries,
    ),
}

reranker_history = train_stage(
    reranker,
    reranker_loss,
    evaluate_reranker,
    RERANKER_EPOCHS,
    "best_reranker.pt",
    baseline=baseline,
)

In [ ]:
selected = torch.load(
    OUTPUT_DIR / "best_reranker.pt",
    map_location="cpu",
    weights_only=True,
)

print(
    "Selected reranker epoch:",
    selected["epoch"],
)

print(
    "Selected Dev result:",
    selected["result"],
)

RERANKER_WEIGHT = selected["result"]["reranker_weight"]

del reranker
gc.collect()

if use_amp:
    torch.cuda.empty_cache()


# Generate test rankings with the best retriever.
retriever.to(device).eval()

test_neural = neural_rankings(
    retriever,
    "test",
)

lexical["test"] = lexical_rankings("test")

shortlists["test"] = fuse_rankings(
    test_neural,
    lexical["test"],
    NEURAL_WEIGHT,
)

retriever.cpu()

if use_amp:
    torch.cuda.empty_cache()


if RERANKER_WEIGHT == 0:
    print(
        "Using hybrid baseline: reranking did not "
        "improve validation F1."
    )

    final_rankings = shortlists["test"]

else:
    reranker, _ = load_phase7(
        OUTPUT_DIR / "best_reranker.pt"
    )

    reranked = rerank(
        reranker,
        "test",
    )

    final_rankings = combine_reranker(
        reranked,
        "test",
        RERANKER_WEIGHT,
    )


predictions = {
    query_id: ids[:7]
    for query_id, ids in final_rankings.items()
}

assert set(predictions) == set(test_queries)

for ids in predictions.values():
    assert len(ids) == len(set(ids)) == 7
    assert set(ids) <= set(test_candidates)


prediction_path = OUTPUT_DIR / "prediction.json"

prediction_path.write_text(
    json.dumps(predictions, indent=2),
    encoding="utf-8",
)

submission_path = OUTPUT_DIR / "submission.zip"

with zipfile.ZipFile(
    submission_path,
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(
        prediction_path,
        arcname="prediction.json",
    )


shutil.copy2(
    MODEL_DIR / "tokenizer.json",
    OUTPUT_DIR / "tokenizer.json",
)

settings = {
    "base_model": str(MODEL_DIR / "model.pt"),
    "seed": SEED,
    "neural_weight": NEURAL_WEIGHT,
    "reranker_weight": RERANKER_WEIGHT,
    "retrieve_top_k": RETRIEVE_TOP_K,
    "rerank_top_k": RERANK_TOP_K,
    "rrf_k": RRF_K,
    "retrieval_length": RETRIEVAL_LENGTH,
    "max_chunks": MAX_CHUNKS,
    "chunk_overlap": CHUNK_OVERLAP,
    "pair_length": PAIR_LENGTH,
    "passage_tokens": PASSAGE_TOKENS,
    "passage_overlap": PASSAGE_OVERLAP,
    "passage_views": PASSAGE_VIEWS,
}

(OUTPUT_DIR / "settings.json").write_text(
    json.dumps(settings, indent=2)
)

print("Submission:", submission_path)

# **Knowledge Distillation**

In [ ]:
from dataclasses import asdict, replace
from torch.utils.checkpoint import checkpoint as checkpoint_block

TEACHER_DIR = OUTPUT_DIR
KD_DIR = Path("/kaggle/working/phase7_model1_kd")
KD_DIR.mkdir(parents=True, exist_ok=True)

KD_EPOCHS = 3
KD_LR = 1e-4
KD_WEIGHT = 0.5
KD_TEMPERATURE = 2.0
KD_PATIENCE = 2

KD_BATCH_SIZE = QUERY_BATCH_SIZE

KD_RETRIEVER_CHECKPOINTING = True
KD_RERANKER_CHECKPOINTING = False

training_ids = [
    q for q, row in train_queries.items()
    if 0 < len(row["relevant_candidates"]) < len(train_candidates)
]

train_candidate_ids = list(train_candidates)

# Release previously loaded models.
if "retriever" in globals():
    del retriever

if "reranker" in globals():
    del reranker

gc.collect()

if use_amp:
    torch.cuda.empty_cache()

In [ ]:
class KDStudent(nn.Module):
    def __init__(self, cfg, task):
        super().__init__()

        base = DecoderLM(cfg)
        patch_attention(base)

        self.cfg = cfg
        self.task = task

        self.tok_emb = base.tok_emb
        self.blocks = base.blocks
        self.ln_f = base.ln_f

        # Freeze the original weights.
        self.requires_grad_(False)

        # Train LoRA adapters.
        for block in self.blocks:
            block.attn.q_proj = LoRALinear(block.attn.q_proj)
            block.attn.v_proj = LoRALinear(block.attn.v_proj)

        output_size = PROJECTION_DIM if task == "retriever" else 1
        self.head = nn.Linear(cfg.d_model, output_size, bias=False)

    def forward(self, ids, lengths):
        x = self.tok_emb(ids)

        use_checkpoint = (
            KD_RETRIEVER_CHECKPOINTING
            if self.task == "retriever"
            else KD_RERANKER_CHECKPOINTING
        )

        for block in self.blocks:
            if self.training and use_checkpoint:
                x = checkpoint_block(block, x, use_reentrant=False)
            else:
                x = block(x)

        rows = torch.arange(len(ids), device=ids.device)
        return self.ln_f(x[rows, lengths - 1])


def make_student(teacher):
    teacher_depth = len(teacher.blocks)

    if teacher_depth < 2:
        raise ValueError("The teacher must have at least two blocks.")

    student_depth = teacher_depth // 2

    selected_layers = np.linspace(
        0,
        teacher_depth - 1,
        student_depth,
        dtype=int,
    ).tolist()

    cfg = replace(teacher.cfg, n_layers=student_depth)
    student = KDStudent(cfg, teacher.task)

    student.tok_emb.load_state_dict(teacher.tok_emb.state_dict())
    student.ln_f.load_state_dict(teacher.ln_f.state_dict())
    student.head.load_state_dict(teacher.head.state_dict())

    for block, index in zip(student.blocks, selected_layers):
        block.load_state_dict(teacher.blocks[index].state_dict())

    print(
        f"{teacher.task}: "
        f"{teacher_depth} teacher blocks -> {student_depth} student blocks"
    )

    return student.to(device)


def kd_settings():
    return {
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "projection_dim": PROJECTION_DIM,
    }


def save_student(model, path, epoch, result):
    saved = {
        "model_config": asdict(model.cfg),
        "task": model.task,
        "settings": kd_settings(),
        "epoch": epoch,
        "result": result,
        "model_state_dict": {
            name: value.detach().cpu().clone()
            for name, value in model.state_dict().items()
        },
    }

    temporary = path.with_suffix(".tmp")
    torch.save(saved, temporary)
    temporary.replace(path)


def load_student(path):
    saved = torch.load(path, map_location="cpu", weights_only=True)

    assert saved["settings"] == kd_settings(), "Use the saved LoRA settings."

    model = KDStudent(
        LMConfig(**saved["model_config"]),
        saved["task"],
    )

    model.load_state_dict(saved["model_state_dict"])
    return model.to(device).eval(), saved

In [ ]:
def distillation_loss(student_scores, teacher_scores):
    temperature = KD_TEMPERATURE

    return F.kl_div(
        F.log_softmax(student_scores.float() / temperature, dim=-1),
        F.softmax(teacher_scores.float() / temperature, dim=-1),
        reduction="batchmean",
    ) * temperature**2


def train_kd(model, loss_function, evaluate, filename):
    parameters = [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ]

    optimizer = torch.optim.AdamW(
        parameters,
        lr=KD_LR,
        weight_decay=0.01,
        fused=device.type == "cuda",
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=use_amp and amp_dtype == torch.float16,
    )

    best_f1 = -1.0
    bad_epochs = 0

    for epoch in range(1, KD_EPOCHS + 1):
        model.train()

        order = training_ids.copy()
        random.shuffle(order)

        total_loss = 0.0
        batches = math.ceil(len(order) / KD_BATCH_SIZE)

        for step, start in enumerate(
            range(0, len(order), KD_BATCH_SIZE), 1
        ):
            query_ids = order[start:start + KD_BATCH_SIZE]
            optimizer.zero_grad(set_to_none=True)

            # Accumulate reranker gradients one query at a time
            # to avoid retaining every query's computation graph.
            micro_batches = (
                [[q] for q in query_ids]
                if model.task == "reranker"
                else [query_ids]
            )

            batch_loss = torch.zeros((), device=device)

            for micro_ids in micro_batches:
                with amp_context():
                    loss = loss_function(model, micro_ids)

                weight = len(micro_ids) / len(query_ids)
                scaler.scale(loss * weight).backward()
                batch_loss += loss.detach().float() * weight

            if not torch.isfinite(batch_loss):
                raise RuntimeError("Non-finite KD loss.")

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(parameters, 1.0)
            scaler.step(optimizer)
            scaler.update()

            total_loss += batch_loss.item() * len(query_ids)

            if step % 100 == 0 or step == batches:
                print(
                    f"{model.task} KD | Epoch {epoch} | "
                    f"Batch {step}/{batches} | "
                    f"Loss: {batch_loss.item():.4f}",
                    flush=True,
                )

        model.eval()

        with torch.inference_mode():
            result = evaluate(model)

        print({
            "epoch": epoch,
            "train_loss": total_loss / len(order),
            **result,
        })

        if result["f1"] > best_f1:
            best_f1 = result["f1"]
            bad_epochs = 0

            save_student(
                model,
                KD_DIR / filename,
                epoch,
                result,
            )
        else:
            bad_epochs += 1

            if bad_epochs >= KD_PATIENCE:
                break

In [ ]:
teacher, _ = load_phase7(TEACHER_DIR / "best_retriever.pt")
teacher.eval().requires_grad_(False)

teacher_q_ids, teacher_q_vectors = make_index(
    teacher, chunks["train"][0]
)

teacher_c_ids, teacher_c_vectors = make_index(
    teacher, chunks["train"][1]
)

teacher_q_rows = {
    query_id: index
    for index, query_id in enumerate(teacher_q_ids)
}

teacher_c_rows = {
    candidate_id: index
    for index, candidate_id in enumerate(teacher_c_ids)
}

teacher.cpu()
student_retriever = make_student(teacher)

del teacher
gc.collect()

if use_amp:
    torch.cuda.empty_cache()


def retriever_kd_loss(model, query_ids):
    candidates = []

    for query_id in query_ids:
        sampled, _ = sample_candidates(query_id, lexical["train"])
        candidates.extend(sampled)

    candidates = list(dict.fromkeys(candidates))

    query_vectors = encode_documents(
        model,
        [chunks["train"][0][q] for q in query_ids],
    )

    candidate_vectors = encode_documents(
        model,
        [chunks["train"][1][c] for c in candidates],
    )

    student_scores = (
        query_vectors @ candidate_vectors.T
    ) / TEMPERATURE

    teacher_queries = teacher_q_vectors[
        [teacher_q_rows[q] for q in query_ids]
    ].to(device)

    teacher_candidates = teacher_c_vectors[
        [teacher_c_rows[c] for c in candidates]
    ].to(device)

    teacher_scores = (
        teacher_queries @ teacher_candidates.T
    ) / TEMPERATURE

    positive = torch.tensor(
        [
            [
                c in train_queries[q]["relevant_candidates"]
                for c in candidates
            ]
            for q in query_ids
        ],
        device=device,
        dtype=torch.bool,
    )

    log_probs = F.log_softmax(student_scores.float(), dim=-1)

    supervised_loss = -(log_probs * positive).sum(dim=1)
    supervised_loss = (
        supervised_loss / positive.sum(dim=1)
    ).mean()

    kd_loss = distillation_loss(student_scores, teacher_scores)

    return (
        (1 - KD_WEIGHT) * supervised_loss
        + KD_WEIGHT * kd_loss
    )


train_kd(
    student_retriever,
    retriever_kd_loss,
    evaluate_retriever,
    "best_student_retriever.pt",
)

del student_retriever
gc.collect()

if use_amp:
    torch.cuda.empty_cache()

student_retriever, _ = load_student(
    KD_DIR / "best_student_retriever.pt"
)

In [ ]:
dev_neural = neural_rankings(student_retriever, "dev")

results = []

for weight in NEURAL_WEIGHTS:
    rankings = fuse_rankings(
        dev_neural,
        lexical["dev"],
        weight,
    )

    results.append({
        "neural_weight": weight,
        **retrieval_scores(rankings, dev_queries),
    })

best_hybrid = max(results, key=lambda row: row["f1"])
NEURAL_WEIGHT = best_hybrid["neural_weight"]

print("Student hybrid validation:", best_hybrid)

shortlists = {
    "dev": fuse_rankings(
        dev_neural,
        lexical["dev"],
        NEURAL_WEIGHT,
    )
}

train_neural = neural_rankings(student_retriever, "train")

shortlists["train"] = fuse_rankings(
    train_neural,
    lexical["train"],
    NEURAL_WEIGHT,
)

student_retriever.cpu()

if use_amp:
    torch.cuda.empty_cache()

In [ ]:
teacher, _ = load_phase7(TEACHER_DIR / "best_reranker.pt")
teacher.eval().requires_grad_(False)

kd_pairs = {}

with torch.inference_mode():
    for step, query_id in enumerate(training_ids, 1):
        candidates, n_positive = sample_candidates(
            query_id,
            shortlists["train"],
        )

        with amp_context():
            scores = score_pairs(
                teacher,
                "train",
                query_id,
                candidates,
                EVAL_BATCH_SIZE,
            )

        kd_pairs[query_id] = {
            "candidates": candidates,
            "n_positive": n_positive,
            "teacher_scores": scores.float().cpu(),
        }

        if step % 100 == 0 or step == len(training_ids):
            print(
                f"Teacher scores cached: {step}/{len(training_ids)}",
                flush=True,
            )

teacher.cpu()
student_reranker = make_student(teacher)

del teacher
gc.collect()

if use_amp:
    torch.cuda.empty_cache()


def reranker_kd_loss(model, query_ids):
    losses = []

    for query_id in query_ids:
        row = kd_pairs[query_id]

        student_scores = score_pairs(
            model,
            "train",
            query_id,
            row["candidates"],
            PAIR_BATCH_SIZE,
        )

        teacher_scores = row["teacher_scores"].to(device)

        n_positive = row["n_positive"]

        positive_scores = student_scores[:n_positive]
        negative_scores = student_scores[n_positive:]

        supervised_loss = F.softplus(
            negative_scores[None, :] - positive_scores[:, None]
        ).mean()

        kd_loss = distillation_loss(
            student_scores[None, :],
            teacher_scores[None, :],
        )

        losses.append(
            (1 - KD_WEIGHT) * supervised_loss
            + KD_WEIGHT * kd_loss
        )

    return torch.stack(losses).mean()


train_kd(
    student_reranker,
    reranker_kd_loss,
    evaluate_reranker,
    "best_student_reranker.pt",
)

del student_reranker
gc.collect()

if use_amp:
    torch.cuda.empty_cache()

In [ ]:
student_retriever.to(device).eval()

test_neural = neural_rankings(student_retriever, "test")
test_lexical = lexical_rankings("test")

shortlists["test"] = fuse_rankings(
    test_neural,
    test_lexical,
    NEURAL_WEIGHT,
)

student_retriever.cpu()

if use_amp:
    torch.cuda.empty_cache()

student_reranker, saved = load_student(
    KD_DIR / "best_student_reranker.pt"
)

RERANKER_WEIGHT = saved["result"]["reranker_weight"]

if RERANKER_WEIGHT == 0:
    final_rankings = shortlists["test"]
else:
    reranked = rerank(student_reranker, "test")

    final_rankings = combine_reranker(
        reranked,
        "test",
        RERANKER_WEIGHT,
    )

predictions = {
    query_id: candidates[:7]
    for query_id, candidates in final_rankings.items()
}

prediction_path = KD_DIR / "prediction.json"

prediction_path.write_text(
    json.dumps(predictions, indent=2),
    encoding="utf-8",
)

with zipfile.ZipFile(
    KD_DIR / "submission.zip",
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(prediction_path, arcname="prediction.json")

(KD_DIR / "fusion_weights.json").write_text(
    json.dumps({
        "neural_weight": NEURAL_WEIGHT,
        "reranker_weight": RERANKER_WEIGHT,
    }, indent=2),
    encoding="utf-8",
)

print("Submission:", KD_DIR / "submission.zip")